In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import json
import numpy as np
from sources.MKP import MultipleKnapsackProblem
from sources.MKPsolver import MKPQAOASolver
from matplotlib import pyplot as plt

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)


In [4]:

# Load instances
with open('data/small_instances.json', 'r') as f:
    instances = json.load(f)

In [6]:

# Test scenario_1
scenario = instances['scenario_1']
mkp = MultipleKnapsackProblem(
    article_reward=scenario['article_reward'],
    article_weight=scenario['article_weight'],
    knapsack_capacity=scenario['knapsack_capacity']
)


In [ ]:
# Test QAOA solver for one problem instance

print("\n" + "="*60)
print("TESTING QAOA SOLVER")
print("="*60)

print("\nTesting QAOA Solver on scenario_1...")
print("This may take a few minutes...\n")

# Create QAOA solver with p=1 layer
qaoa_solver = MKPQAOASolver(mkp, p=2, use_slack=False, cont_slack=False, unbalanced_penalization=True)

# Solve
best_bitstring, best_value, optimal_params, counts, energy = qaoa_solver.solve(shots=100, maxiter=50)

print(f"QAOA solution (bitstring): {best_bitstring}")
print(f"QAOA value: {best_value}")
print(f"Expected optimal: {-scenario['optimal_solution']}")
print(f"Optimal parameters (gamma, beta): {optimal_params}")
print(f"\nApproximation ratio: {best_value / -scenario['optimal_solution']:.2%}")
print(counts)



TESTING QAOA SOLVER

Testing QAOA Solver on scenario_1...
This may take a few minutes...

Initial energy:  36.33
Optimizing QAOA (use_slack=False, p=2)...
Energy after parameter tuning: 24.580000000000002
QAOA solution (bitstring): 1000
QAOA value: -4.0
Expected optimal: -4
Optimal parameters (gamma, beta): [1.42087584 1.80889421 0.36442357 0.79915307]

Approximation ratio: 100.00%
{'0000': 17.0, '0001': 24.0, '0010': 12.0, '0011': 15.0, '0100': 3.0, '0101': 5.0, '0110': 1.0, '0111': 6.0, '1000': 4.0, '1001': 1.0, '1010': 3.0, '1100': 1.0, '1101': 1.0, '1110': 2.0, '1111': 5.0}


In [8]:
# helper function to get results from QAOA counts

def analyze_counts(mkp:MultipleKnapsackProblem, counts, shots, opt_val):
    # feasiblity
    num_feasible = 0
    for b in counts:
        if mkp.evaluate_bitstring(b) <= 0:
            num_feasible += counts[b]
    feasiblity_ratio = num_feasible/shots
    # percentage of optimal solutions
    num_optimal = 0
    for b in counts:
        if mkp.evaluate_bitstring(b) == -opt_val:
            num_feasible += counts[b]
    optimality_ratio = num_optimal/shots
    # percentage of 90% optimal solutions
    num_90_optimal = 0
    for b in counts:
        if mkp.evaluate_bitstring(b) <= 0.9*(-opt_val):
            num_90_optimal += counts[b]
    almost_optimal_ratio = num_90_optimal/shots

    return feasiblity_ratio, optimality_ratio, almost_optimal_ratio




In [9]:
# set up overall test structure for instances 0 to 9

shots = 1024
p=5

results_standard = {}
results_no_slack = {}
results_unbalanced_penalization = {}
results_cont_slack = {}


for i in range(10):
    print(f"Processing scenario {i}...")
    scenario_data = instances[f"scenario_{i}"]
    
    # Create MKP instance
    mkp_instance = MultipleKnapsackProblem(
        article_reward=scenario_data['article_reward'],
        article_weight=scenario_data['article_weight'],
        knapsack_capacity=scenario_data['knapsack_capacity']
    )
    
    # Classical solution
    classical_val = -scenario_data["optimal_solution"]
    
    # QAOA solution with 4 different settings, 5 layers
    qaoa_standard = MKPQAOASolver(mkp_instance, p=p, use_slack=True)

    qaoa_no_slack = MKPQAOASolver(mkp_instance, p=p, use_slack=False)
    qaoa_cont_slack = MKPQAOASolver(mkp_instance, p=p, cont_slack=True)
    qaoa_unbalanced_penalization = MKPQAOASolver(mkp_instance, p=p, unbalanced_penalization=True)
    # do 20 runs
    results_standard[i]={}
    results_no_slack[i]={}
    results_cont_slack[i]={}
    results_unbalanced_penalization[i]={}
    for r in range(20):
        print(f"Run {r}")
        qaoa_bitstring, qaoa_val, qaoa_params, counts, energy = qaoa_standard.solve(shots=shots, maxiter=30)
        feasibility_ratio, optimality_ratio, opt_90_ratio = analyze_counts(mkp_instance, counts, shots, classical_val)
        results_standard[i][r] = {
            'qaoa_value': qaoa_val,
            'optimal_value': scenario_data['optimal_solution'],
            'qaoa_bitstring': qaoa_bitstring,
            'qaoa_energy': energy,
            'feasibility': feasibility_ratio,
            'optimality_percentage': optimality_ratio,
            'opt_90_percentage': opt_90_ratio
        }
        qaoa_bitstring, qaoa_val, qaoa_params, counts, energy = qaoa_no_slack.solve(shots=shots, maxiter=30)
        feasibility_ratio, optimality_ratio, opt_90_ratio = analyze_counts(mkp_instance, counts, shots, classical_val)
        results_no_slack[i][r] = {
            'qaoa_value': qaoa_val,
            'optimal_value': scenario_data['optimal_solution'],
            'qaoa_bitstring': qaoa_bitstring,
            'qaoa_energy': energy,
            'feasibility': feasibility_ratio,
            'optimality_percentage': optimality_ratio,
            'opt_90_percentage': opt_90_ratio

        }
        qaoa_bitstring, qaoa_val, qaoa_params, counts, energy = qaoa_cont_slack.solve(shots=shots, maxiter=30)
        feasibility_ratio, optimality_ratio, opt_90_ratio = analyze_counts(mkp_instance, counts, shots, classical_val)
        s_values = qaoa_params[2*p:]
        results_cont_slack[i][r] = {
            'qaoa_value': qaoa_val,
            'optimal_value': scenario_data['optimal_solution'],
            'qaoa_bitstring': qaoa_bitstring,
            'qaoa_energy': energy,
            'feasibility': feasibility_ratio,
            'optimality_percentage': optimality_ratio,
            's_values': s_values,
            'opt_90_percentage': opt_90_ratio
        }
        qaoa_bitstring, qaoa_val, qaoa_params, counts, energy = qaoa_cont_slack.solve(shots=shots, maxiter=30)
        feasibility_ratio, optimality_ratio, opt_90_ratio = analyze_counts(mkp_instance, counts, shots, classical_val)
        results_unbalanced_penalization[i][r] = {
            'qaoa_value': qaoa_val,
            'optimal_value': scenario_data['optimal_solution'],
            'qaoa_bitstring': qaoa_bitstring,
            'qaoa_energy': energy,
            'feasibility': feasibility_ratio,
            'optimality_percentage': optimality_ratio,
            'opt_90_percentage': opt_90_ratio
        }

Processing scenario 0...
Run 0
Initial energy:  -8.0283203125
Optimizing QAOA (use_slack=True, p=5)...
Energy after parameter tuning: -10.6767578125
Initial energy:  -16.4931640625
Optimizing QAOA (use_slack=False, p=5)...
Energy after parameter tuning: -17.775390625
Initial energy:  636.6623467030965
Optimizing QAOA (use_slack=False, p=5)...
Energy after parameter tuning: 52.8475962750249
Initial energy:  439.808967095878
Optimizing QAOA (use_slack=False, p=5)...
Energy after parameter tuning: 223.3354387114762
Run 1
Initial energy:  -6.6396484375
Optimizing QAOA (use_slack=True, p=5)...
Energy after parameter tuning: -10.3046875
Initial energy:  -1.8564453125
Optimizing QAOA (use_slack=False, p=5)...
Energy after parameter tuning: -16.544921875
Initial energy:  533.9312272074383
Optimizing QAOA (use_slack=False, p=5)...
Energy after parameter tuning: 96.89840772377002
Initial energy:  359.2943864541983
Optimizing QAOA (use_slack=False, p=5)...
Energy after parameter tuning: 292.56674

KeyboardInterrupt: 

In [ ]:
# Save results

with open("results/standard_p=5_s=1024.json", "w") as outfile:
    json.dump(results_standard, outfile, indent=4, cls=NumpyEncoder)
with open("results/no_slack_p=5_s=1024.json", "w") as outfile:
    json.dump(results_no_slack, outfile, indent=4, cls=NumpyEncoder)
with open("results/cont_slack_p=5_s=1024.json", "w") as outfile:
    json.dump(results_cont_slack, outfile, indent=4, cls=NumpyEncoder)
with open("results/unbalanced_p=5_s=1024.json", "w") as outfile:
    json.dump(results_unbalanced_penalization, outfile, indent=4, cls=NumpyEncoder)

NameError: name 'results_standard' is not defined